# CFG feature sets and ML prediction of logtau

This notebook loads CFG-derived descriptors, builds CFG-only and CFG-plus-temperature inputs, tunes multiple regression models with five-fold cross-validation and GridSearchCV, and saves evaluation results.

In [4]:
# Imports
import os
import re
import json
import shutil
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import skew, kurtosis
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR


In [5]:
# Configuration
metadata_file = Path(r"C:\Users\Admin\Downloads\cfg data\mapped_cfg_and_metadata\metadata_all_materials.csv")
output_dir = Path("ML_output_cfg_logtau")
output_dir.mkdir(parents=True, exist_ok=True)
# Required metadata columns:
# material, cfg_path_mapped, cfg_file_mapped, T_cfg, logtau
TARGET_COL = "logtau"
TEMP_COL = "temperature"
RANDOM_STATE = 42
N_SPLITS = 5
TEST_SIZE = 0.2
# Coordination-number cutoff in angstroms.
CUT_OFF_CN = 3.6
# Distribution-descriptor bins.
CN_BINS = list(range(6, 19))
HIST_BINS = 12
PERCENTILES = [5, 10, 25, 50, 75, 90, 95]


In [6]:
features_df = pd.read_csv(r"C:\Users\Admin\Downloads\cfg data\ml_features.csv")
features_df.head()

,n_atoms,element_id_mean,element_id_std,element_id_min,element_id_max,element_id_median,element_id_p05,element_id_p25,element_id_p75,element_id_p95,...,csym_p25,csym_p75,csym_p95,csym_skew,csym_kurtosis,material,cfg_file,cfg_path,temperature,logtau
0,4000,1.54925,0.575391,1.0,3.0,2.0,1.0,1.0,2.0,2.0,...,7.098170,10.437425,12.480505,-0.380736,-0.065228,Zr46Cu46Al8,Zr46Cu46Al8_638.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,638.0,15.722426
1,4000,1.54925,0.575391,1.0,3.0,2.0,1.0,1.0,2.0,2.0,...,7.124150,10.394525,12.464310,-0.324969,0.034584,Zr46Cu46Al8,Zr46Cu46Al8_661.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,661.0,10.618194
2,4000,1.54925,0.575391,1.0,3.0,2.0,1.0,1.0,2.0,2.0,...,7.181178,10.411450,12.390040,-0.392601,0.055420,Zr46Cu46Al8,Zr46Cu46Al8_673.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,673.0,8.450109
3,4000,1.54925,0.575391,1.0,3.0,2.0,1.0,1.0,2.0,2.0,...,7.264575,10.481300,12.471135,-0.348448,-0.047570,Zr46Cu46Al8,Zr46Cu46Al8_678.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,678.0,7.564629
4,4000,1.54925,0.575391,1.0,3.0,2.0,1.0,1.0,2.0,2.0,...,7.209983,10.479425,12.642630,-0.345779,0.026625,Zr46Cu46Al8,Zr46Cu46Al8_680.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,680.0,7.218741


In [7]:
import os
from pathlib import Path
# Build the two input feature sets
# Ensure that the output directory is available.
output_dir = Path("ML_output_cfg_logtau")
output_dir.mkdir(parents=True, exist_ok=True)
id_cols = ["material", "cfg_file", "cfg_path", "n_atoms"]
non_feature_cols = id_cols + [TARGET_COL]
feature_cols_with_temp = [c for c in features_df.columns if c not in non_feature_cols]
feature_cols_cfg_only = [c for c in feature_cols_with_temp if c != TEMP_COL]
feature_sets = {
    "cfg_only": feature_cols_cfg_only,
    "cfg_plus_temperature": feature_cols_with_temp,
}
for name, cols in feature_sets.items():
    out = features_df[id_cols + cols + [TARGET_COL]].copy()
    out.to_csv(output_dir / f"dataset_{name}.csv", index=False)
    print(name, out.shape)


cfg_only (126, 82)
cfg_plus_temperature (126, 83)


In [8]:
print("Total columns cfg_only:", len(id_cols + feature_cols_cfg_only + [TARGET_COL]))
print("Number of actual features cfg_only:", len(feature_cols_cfg_only))
print("Is n_atoms feature?", "n_atoms" in feature_cols_cfg_only)
print("Is n_atoms id?", "n_atoms" in id_cols)

Total columns cfg_only: 82
Number of actual features cfg_only: 77
Is n_atoms feature? False
Is n_atoms id? True


In [9]:
# Models and hyperparameter grids
def get_models_and_grids():
    models = {
        "Extra Trees": ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        "KNN": KNeighborsRegressor(),
        "SVR": SVR(),
        "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    }
    grids = {
        "Extra Trees": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
        },
        "Random Forest": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
        },
        "KNN": {
            "model__n_neighbors": [3, 5, 7],
            "model__weights": ["uniform", "distance"],
        },
        "SVR": {
            "model__C": [1, 10, 100],
            "model__epsilon": [0.01, 0.1],
            "model__kernel": ["rbf"],
        },
        "Decision Tree": {
            "model__max_depth": [None, 3, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
        },
        "Gradient Boosting": {
            "model__n_estimators": [100, 300],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [2, 3],
        },
            "XGBoost": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [2, 3, 5],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
            }
        }
    return models, grids
def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


In [10]:
# Evaluation metrics
def calc_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": np.sqrt(mse),
        "mae": mean_absolute_error(y_true, y_pred),
    }
def clean_params(params):
    return {k.replace("model__", ""): v for k, v in params.items()}


In [11]:
# Five-fold CV, GridSearchCV, and hold-out evaluation
def run_ml_for_feature_set(features_df, feature_cols, set_name):
    X = features_df[feature_cols].copy()
    y = features_df[TARGET_COL].astype(float).values
    meta = features_df[["material", "cfg_file", "cfg_path", TEMP_COL]].copy()
    X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
        X, y, meta, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    models, grids = get_models_and_grids()
    outer_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_records = []
    summary_records = []
    fitted_models = {}
    performance_df = pd.DataFrame(index=np.arange(len(y_test)))
    performance_df[("Info", "material")] = meta_test["material"].values
    performance_df[("Info", "cfg_file")] = meta_test["cfg_file"].values
    performance_df[("Info", "temperature")] = meta_test[TEMP_COL].values
    for model_name, model in models.items():
        print(f"[{set_name}] Running {model_name}...")
        pipe = make_pipeline(model)
        grid = grids[model_name]
        y_test_pred_folds = []
        fold_best_params = []
        for fold, (tr_idx, val_idx) in enumerate(outer_cv.split(X_train), start=1):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]
            search = GridSearchCV(
                pipe, grid, cv=3, scoring="r2", n_jobs=-1, error_score="raise"
            )
            search.fit(X_tr, y_tr)
            best = search.best_estimator_
            pred_tr = best.predict(X_tr)
            pred_val = best.predict(X_val)
            m_tr = calc_metrics(y_tr, pred_tr)
            m_val = calc_metrics(y_val, pred_val)
            fold_records.append({
                "feature_set": set_name,
                "model": model_name,
                "fold": fold,
                "best_params": json.dumps(clean_params(search.best_params_)),
                "train_r2": m_tr["r2"],
                "train_rmse": m_tr["rmse"],
                "train_mae": m_tr["mae"],
                "val_r2": m_val["r2"],
                "val_rmse": m_val["rmse"],
                "val_mae": m_val["mae"],
            })
            fold_best_params.append(search.best_params_)
            y_test_pred_folds.append(best.predict(X_test))
        # Final search on the full training set.
        final_search = GridSearchCV(
            pipe, grid, cv=5, scoring="r2", n_jobs=-1, error_score="raise"
        )
        final_search.fit(X_train, y_train)
        final_model = final_search.best_estimator_
        fitted_models[model_name] = final_model
        y_pred_train = final_model.predict(X_train)
        y_pred_test = final_model.predict(X_test)
        train_m = calc_metrics(y_train, y_pred_train)
        test_m = calc_metrics(y_test, y_pred_test)
        summary_records.append({
            "feature_set": set_name,
            "model": model_name,
            "best_params": json.dumps(clean_params(final_search.best_params_)),
            "train_r2": train_m["r2"],
            "train_rmse": train_m["rmse"],
            "train_mae": train_m["mae"],
            "test_r2": test_m["r2"],
            "test_rmse": test_m["rmse"],
            "test_mae": test_m["mae"],
        })
        performance_df[(model_name, "y_true")] = y_test
        performance_df[(model_name, "y_pred")] = y_pred_test
    fold_df = pd.DataFrame(fold_records)
    summary_df = pd.DataFrame(summary_records).sort_values(
        ["test_r2", "test_rmse", "test_mae"], ascending=[False, True, True]
    )
    # Save metrics and test predictions.
    fold_df.to_csv(output_dir / f"metrics_folds_{set_name}.csv", index=False)
    summary_df.to_csv(output_dir / f"metrics_summary_{set_name}.csv", index=False)
    performance_df.to_csv(output_dir / f"test_performance_{set_name}.csv", index=False)
    return fold_df, summary_df, performance_df, fitted_models


In [12]:
# Run both input feature sets
# Ensure that shared variables are available in this cell.
RANDOM_STATE = 42
TEST_SIZE = 0.2
all_fold_metrics = []
all_summary = []
results = {}
for set_name, cols in feature_sets.items():
    fold_df, summary_df, perf_df, fitted_models = run_ml_for_feature_set(features_df, cols, set_name)
    all_fold_metrics.append(fold_df)
    all_summary.append(summary_df)
    results[set_name] = {
        "fold": fold_df,
        "summary": summary_df,
        "performance": perf_df,
        "fitted_models": fitted_models,
    }
all_fold_metrics = pd.concat(all_fold_metrics, ignore_index=True)
all_summary = pd.concat(all_summary, ignore_index=True)
all_fold_metrics.to_csv(output_dir / "metrics_folds_all_feature_sets.csv", index=False)
all_summary.to_csv(output_dir / "metrics_summary_all_feature_sets.csv", index=False)
print("Done. Saved outputs to:", output_dir)
all_summary


[cfg_only] Running Extra Trees...
[cfg_only] Running Random Forest...
[cfg_only] Running KNN...
[cfg_only] Running SVR...
[cfg_only] Running Decision Tree...
[cfg_only] Running Gradient Boosting...
[cfg_plus_temperature] Running Extra Trees...
[cfg_plus_temperature] Running Random Forest...
[cfg_plus_temperature] Running KNN...
[cfg_plus_temperature] Running SVR...
[cfg_plus_temperature] Running Decision Tree...
[cfg_plus_temperature] Running Gradient Boosting...
Done. Saved outputs to: ML_output_cfg_logtau


,feature_set,model,best_params,train_r2,train_rmse,train_mae,test_r2,test_rmse,test_mae
0,cfg_only,Gradient Boosting,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",1.000000,1.092501e-03,8.974038e-04,0.912037,2.505457,1.827224
1,cfg_only,Extra Trees,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.999997,1.179484e-02,7.635458e-03,0.873516,3.004374,2.033277
2,cfg_only,SVR,"{""C"": 100, ""epsilon"": 0.01, ""kernel"": ""rbf""}",0.999998,1.009096e-02,1.008915e-02,0.839745,3.381755,2.404504
3,cfg_only,Random Forest,"{""max_depth"": 5, ""min_samples_leaf"": 1, ""n_est...",0.974737,1.180084e+00,8.247345e-01,0.826863,3.515052,2.394816
4,cfg_only,KNN,"{""n_neighbors"": 3, ""weights"": ""distance""}",1.000000,1.266411e-07,5.406851e-08,0.792349,3.849497,2.722036
5,cfg_only,Decision Tree,"{""max_depth"": 3, ""min_samples_leaf"": 1}",0.900614,2.340624e+00,1.567943e+00,0.767353,4.074600,3.287665
6,cfg_plus_temperature,Gradient Boosting,"{""learning_rate"": 0.03, ""max_depth"": 2, ""n_est...",0.999313,1.946501e-01,1.474554e-01,0.945233,1.976953,1.173795
7,cfg_plus_temperature,Extra Trees,"{""max_depth"": null, ""min_samples_leaf"": 2, ""n_...",0.999264,2.014892e-01,1.294188e-01,0.936533,2.128196,1.265262
8,cfg_plus_temperature,Random Forest,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.986134,8.742815e-01,5.517836e-01,0.895875,2.725926,1.464978
9,cfg_plus_temperature,SVR,"{""C"": 100, ""epsilon"": 0.01, ""kernel"": ""rbf""}",0.999998,1.002682e-02,1.002471e-02,0.847055,3.303728,2.328718


In [13]:
# Display the best results
summary_view = all_summary.copy()
for c in ["train_r2", "test_r2"]:
    summary_view[c] = summary_view[c] * 100
num_cols = ["train_r2", "train_rmse", "train_mae", "test_r2", "test_rmse", "test_mae"]
summary_view[num_cols] = summary_view[num_cols].round(2)
summary_view.sort_values(["test_r2", "test_rmse", "test_mae"], ascending=[False, True, True])


,feature_set,model,best_params,train_r2,train_rmse,train_mae,test_r2,test_rmse,test_mae
6,cfg_plus_temperature,Gradient Boosting,"{""learning_rate"": 0.03, ""max_depth"": 2, ""n_est...",99.93,0.19,0.15,94.52,1.98,1.17
7,cfg_plus_temperature,Extra Trees,"{""max_depth"": null, ""min_samples_leaf"": 2, ""n_...",99.93,0.20,0.13,93.65,2.13,1.27
0,cfg_only,Gradient Boosting,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",100.00,0.00,0.00,91.20,2.51,1.83
8,cfg_plus_temperature,Random Forest,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",98.61,0.87,0.55,89.59,2.73,1.46
1,cfg_only,Extra Trees,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",100.00,0.01,0.01,87.35,3.00,2.03
9,cfg_plus_temperature,SVR,"{""C"": 100, ""epsilon"": 0.01, ""kernel"": ""rbf""}",100.00,0.01,0.01,84.71,3.30,2.33
2,cfg_only,SVR,"{""C"": 100, ""epsilon"": 0.01, ""kernel"": ""rbf""}",100.00,0.01,0.01,83.97,3.38,2.40
10,cfg_plus_temperature,Decision Tree,"{""max_depth"": 5, ""min_samples_leaf"": 2}",99.37,0.59,0.42,83.73,3.41,2.09
3,cfg_only,Random Forest,"{""max_depth"": 5, ""min_samples_leaf"": 1, ""n_est...",97.47,1.18,0.82,82.69,3.52,2.39
11,cfg_plus_temperature,KNN,"{""n_neighbors"": 3, ""weights"": ""distance""}",100.00,0.00,0.00,79.32,3.84,2.70


## Native feature importance of the best ML model

This section reads the model's built-in `feature_importances_` values. For tree-based models, these values quantify the normalized total reduction in the squared-error splitting criterion attributable to each input feature. No feature is permuted, and the model is not retrained. Individual importance values are also summed into the physical input groups defined in the manuscript.

In [14]:
# Native importance from the best fitted tree model
IMPORTANCE_SET = "cfg_plus_temperature"
# The summary table is already sorted from best to worst test performance.
best_row = (
    results[IMPORTANCE_SET]["summary"]
    .sort_values(
        ["test_r2", "test_rmse", "test_mae"],
        ascending=[False, True, True]
    )
    .iloc[0]
)
best_model_name = best_row["model"]
best_pipeline = results[IMPORTANCE_SET]["fitted_models"][best_model_name]
best_estimator = best_pipeline.named_steps["model"]
if not hasattr(best_estimator, "feature_importances_"):
    raise TypeError(
        f"The best model ({best_model_name}) has no native feature_importances_. "
        "Use a tree-based best model to obtain native feature importance."
    )
feature_names = list(feature_sets[IMPORTANCE_SET])
native_importance = np.asarray(best_estimator.feature_importances_, dtype=float)
if len(feature_names) != len(native_importance):
    raise ValueError(
        f"Feature-name count ({len(feature_names)}) does not match "
        f"the model importance count ({len(native_importance)})."
    )
def assign_physical_group(feature_name):
    if feature_name == TEMP_COL:
        return "Temperature"
    if feature_name.startswith("element_id_"):
        return "Element type"
    if feature_name.startswith(("x_", "y_", "z_")):
        return "Spatial coordinates"
    if feature_name.startswith("energy_"):
        return "Potential energy"
    if feature_name.startswith("force_mag_"):
        return "Force magnitude"
    if feature_name.startswith("csym_"):
        return "Centrosymmetry parameter"
    return "Other"
native_feature_importance_df = pd.DataFrame({
    "feature_set": IMPORTANCE_SET,
    "model": best_model_name,
    "feature": feature_names,
    "feature_group": [assign_physical_group(c) for c in feature_names],
    "native_importance": native_importance,
})
native_feature_importance_df["importance_percent"] = (
    100.0 * native_feature_importance_df["native_importance"]
    / native_feature_importance_df["native_importance"].sum()
)
native_feature_importance_df = native_feature_importance_df.sort_values(
    "native_importance", ascending=False
).reset_index(drop=True)
native_feature_importance_df["importance_rank"] = np.arange(
    1, len(native_feature_importance_df) + 1
)
native_group_importance_df = (
    native_feature_importance_df
    .groupby(["feature_set", "model", "feature_group"], as_index=False)
    .agg(
        n_features=("feature", "count"),
        group_importance=("native_importance", "sum"),
        mean_feature_importance=("native_importance", "mean"),
    )
    .sort_values("group_importance", ascending=False)
    .reset_index(drop=True)
)
native_group_importance_df["group_importance_percent"] = (
    100.0 * native_group_importance_df["group_importance"]
    / native_group_importance_df["group_importance"].sum()
)
native_group_importance_df["importance_rank"] = np.arange(
    1, len(native_group_importance_df) + 1
)
individual_path = output_dir / "native_feature_importance_best_model_individual.csv"
group_path = output_dir / "native_feature_importance_best_model_grouped.csv"
native_feature_importance_df.to_csv(individual_path, index=False)
native_group_importance_df.to_csv(group_path, index=False)
print("Best model:", best_model_name)
print("Number of model inputs:", len(feature_names))
print("Sum of native importance:", native_importance.sum())
print("Saved:", individual_path)
print("Saved:", group_path)
native_group_importance_df

Best model: Gradient Boosting
Number of model inputs: 78
Sum of native importance: 1.0
Saved: ML_output_cfg_logtau\native_feature_importance_best_model_individual.csv
Saved: ML_output_cfg_logtau\native_feature_importance_best_model_grouped.csv


,feature_set,model,feature_group,n_features,group_importance,mean_feature_importance,group_importance_percent,importance_rank
0,cfg_plus_temperature,Gradient Boosting,Temperature,1,0.909524,0.909524,90.952440,1
1,cfg_plus_temperature,Gradient Boosting,Potential energy,11,0.040098,0.003645,4.009751,2
2,cfg_plus_temperature,Gradient Boosting,Spatial coordinates,33,0.029024,0.000880,2.902419,3
3,cfg_plus_temperature,Gradient Boosting,Centrosymmetry parameter,11,0.020564,0.001869,2.056429,4
4,cfg_plus_temperature,Gradient Boosting,Force magnitude,11,0.000790,0.000072,0.078961,5
5,cfg_plus_temperature,Gradient Boosting,Element type,11,0.000000,0.000000,0.000000,6
